### Testing Perplexity

In [4]:
import json

f = open('../../rag_utility/eval_results/short_answers_0shot_1calls_0_0_bm25_dl_nq_test_concise_eval.json')
zero_evals = json.load(f)
f.close()

_k = 3
_ret = 'mt5'
f = open(f'../../rag_utility/eval_results/short_answers_{_k}shot_1calls_1_0_{_ret}_dl_nq_test_concise_eval.json')
k_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/gen_results/short_answers_{_k}shot_1calls_1_0_{_ret}_dl_nq_test_concise.json')
k_gens = json.load(f)
f.close()

# f = open(f'../coherence_eval/log_prob_temp_res/nq_test_{_ret}_12.json')
# k_probs = json.load(f)
# f.close()

f = open(f'../coherence_eval/log_prob_temp_res/full_context/nq_test_{_ret}_{_k}.json')
k_probs = json.load(f)
f.close()


In [9]:
k_gens['test_0']['0']['0']

{'answer': 'Wilhelm Röntgen',
 'prob_seq': -1.7931220531463623,
 'probs': '[np.float32(-0.04275928), np.float32(-2.7060143e-05), np.float32(-0.024742365), np.float32(-0.0013883009), np.float32(-2.0146164e-05), np.float32(-1.1444026e-05), np.float32(-0.0013743727), np.float32(-8.5830325e-06), np.float32(-1.7227905)]'}

In [19]:
import numpy as np

np.array(eval(k_gens['test_0']['0']['0']['probs'])).mean()




-0.19923578

In [20]:
import pandas as pd
import numpy as np

qids_column = list(zero_evals.keys())
zero_f1_column = [zero_evals[qid]['0']['0']['F1'] for qid in qids_column]
k_f1_column = [k_evals[qid]['0']['0']['F1'] for qid in qids_column]
zero_em_column = [zero_evals[qid]['0']['0']['EM'] for qid in qids_column]
k_em_column = [k_evals[qid]['0']['0']['EM'] for qid in qids_column]
posterior_column = [np.array(eval(k_gens[qid]['0']['0']['probs'])).mean() for qid in qids_column]


eval_df = pd.DataFrame(np.array([zero_f1_column, k_f1_column, zero_em_column, k_em_column, posterior_column]).T, columns=['F1(0)', 'F1(k)', 'EM(0)', 'EM(k)', 'posterior'])
eval_df['qid'] = qids_column
eval_df['utility'] = eval_df['F1(k)'] - eval_df['F1(0)']

# try:
#     k_prob_column_dict = {}
#     for _qid, _probs_temp in k_probs.items():
#         k_prob_column_dict.update({_qid: np.mean(list(_probs_temp.values())[:_k])})
#     eval_df = eval_df[eval_df.qid.isin(k_prob_column_dict.keys())]
#     k_prob_column = [k_prob_column_dict[qid] for qid in eval_df.qid.values]
#     eval_df['llm_prob'] = k_prob_column
# except:
#     print('probs are not calculated')
    
# try:
#     k_prob_column = [k_probs[qid]['full'] for qid in qids_column]
#     eval_df['llm_prob'] = k_prob_column
# except:
#     print('probs are not calculated')

try:
    k_prob_column_dict = {}
    for _qid, _probs_temp in k_probs.items():
        k_prob_column_dict.update({_qid: _probs_temp})
    eval_df = eval_df[eval_df.qid.isin(k_prob_column_dict.keys())]
    k_prob_column = [k_prob_column_dict[qid] for qid in eval_df.qid.values]
    eval_df['llm_prob'] = k_prob_column
except:
    print('probs are not calculated')

print(eval_df.shape)
# eval_df.head(20)

(3610, 8)


In [22]:
from scipy import stats
# check posterior for NQ

print('with f1')
print('r', stats.pearsonr(eval_df['F1(k)'].values, eval_df.posterior.values))
print('rho', stats.spearmanr(eval_df['F1(k)'].values, eval_df.posterior.values))
print('tau', stats.kendalltau(eval_df['F1(k)'].values, eval_df.posterior.values))
print('\nwith utility')
print('r', stats.pearsonr(eval_df['utility'].values, eval_df.posterior.values))
print('rho', stats.spearmanr(eval_df['utility'].values, eval_df.posterior.values))
print('tau', stats.kendalltau(eval_df['utility'].values, eval_df.posterior.values))

with f1
r PearsonRResult(statistic=0.3021876969495496, pvalue=4.144172111831524e-77)
rho SignificanceResult(statistic=0.2923707271063318, pvalue=4.4469446632252936e-72)
tau SignificanceResult(statistic=0.22087719042160386, pvalue=4.0105749100656123e-69)

with utility
r PearsonRResult(statistic=0.17211470472994503, pvalue=2.108018717701185e-25)
rho SignificanceResult(statistic=0.16429820027417522, pvalue=2.9099627785954326e-23)
tau SignificanceResult(statistic=0.12101556526085401, pvalue=2.711727998041148e-23)


In [23]:
from scipy import stats
# check context perplexity for NQ

print('with f1')
print('r', stats.pearsonr(eval_df['F1(k)'].values, eval_df.llm_prob.values))
print('rho', stats.spearmanr(eval_df['F1(k)'].values, eval_df.llm_prob.values))
print('tau', stats.kendalltau(eval_df['F1(k)'].values, eval_df.llm_prob.values))
print('\nwith utility')
print('r', stats.pearsonr(eval_df['utility'].values, eval_df.llm_prob.values))
print('rho', stats.spearmanr(eval_df['utility'].values, eval_df.llm_prob.values))
print('tau', stats.kendalltau(eval_df['utility'].values, eval_df.llm_prob.values))

with f1
r PearsonRResult(statistic=0.15939850429094427, pvalue=5.6597276475888735e-22)
rho SignificanceResult(statistic=0.15249546394534924, pvalue=3.165861348106039e-20)
tau SignificanceResult(statistic=0.1149129116670571, pvalue=6.361367535323752e-20)

with utility
r PearsonRResult(statistic=0.06191983244333447, pvalue=0.000197158163718037)
rho SignificanceResult(statistic=0.0644833684118382, pvalue=0.00010572388581496925)
tau SignificanceResult(statistic=0.046776335690365364, pvalue=0.00012230755133291898)


### Test Coherence

In [4]:
from tools import coherence_cal

In [5]:
from tools import matrix_tools

In [6]:
import pickle as pkl

res, _, doc_length_dict = coherence_cal.get_res_and_dicts('nq_test', _ret)


Filename: /mnt/primary/utility_prediction/analysis/tools/coherence_cal.py

Line #    Mem usage    Increment  Occurrences   Line Contents
    53    187.5 MiB    187.5 MiB           1   @profile
    54                                         def get_res_and_dicts(_task, _ret):
    55    187.5 MiB      0.0 MiB           1       material_path = '../../rag_utility'
    56    187.5 MiB      0.0 MiB           1       splitter = SentenceSplitter(language='en')
    57                                         
    58    187.5 MiB      0.0 MiB           1       if(_task == 'dl'):
    59                                                 f = open(f'{material_path}/doc_dicts/msmarco_passage_dict.pkl', 'rb')
    60                                                 dl_19_res = pd.read_csv(f'{material_path}/res/{_ret}_dl_19.csv')
    61                                                 dl_20_res = pd.read_csv(f'{material_path}/res/{_ret}_dl_20.csv')
    62                                                 res =

In [7]:
import math

try:
    
    f = open(f'../coherence_res/bi-directional/nq_test_12_{_ret}.pkl', 'rb')
    matrix_book = pkl.load(f)
    f.close()
    
    w = 10
    coh_dict = coherence_cal.cal_coherence(matrix_book, _k, doc_length_dict, w, math.ceil(w/2), bidirectional=0, distribution='top-heavy')
    coh_df = pd.DataFrame(np.array([list(coh_dict.keys()), list(coh_dict.values())]).T, columns=['qid', 'coherence'])
    print(coh_df.shape)

except:

    print('Not exist the coherence file')
    coh_df = 0

Filename: /mnt/primary/utility_prediction/analysis/tools/coherence_cal.py

Line #    Mem usage    Increment  Occurrences   Line Contents
     8  18860.1 MiB  18860.1 MiB           1   @profile
     9                                         def cal_coherence(_matrix, _k, _doc_length_dict, _window=0, _step=1, bidirectional=0, distribution='uniform'): # 0 for document average, no overlaping # bidirectional: 0: upper; 1: lower; 2:bidirectional
    10  18860.1 MiB      0.0 MiB           1       _coh_dict = {}
    11                                                 
    12  18860.1 MiB      0.0 MiB        3611       for _qid, mtx in _matrix.items():
    13  18860.1 MiB      0.0 MiB        3610           qid = str(_qid)
    14  18860.1 MiB      0.0 MiB        3610           stc_num_in_top_k = np.sum(_doc_length_dict[qid][:_k])
    15  18860.1 MiB      0.0 MiB        3610           cut_mtx = np.matrix(mtx)[:stc_num_in_top_k][:stc_num_in_top_k]
    16                                             

In [8]:
final_df = eval_df.merge(coh_df, on='qid')
final_df.utility = final_df.utility.astype('float')
final_df.coherence = final_df.coherence.astype('float')
print(final_df.shape)
print(final_df['F1(k)'].mean())

from scipy import stats

print('Correlation with utility')
print('r', stats.pearsonr(final_df.coherence.values, final_df.utility.values))
print('rho', stats.spearmanr(final_df.coherence.values, final_df.utility.values))
print('tau', stats.kendalltau(final_df.coherence.values, final_df.utility.values))

print('\nCorrelation with F1')
print('r', stats.pearsonr(final_df.coherence.values, final_df['F1(k)'].values))
print('rho', stats.spearmanr(final_df.coherence.values, final_df['F1(k)'].values))
print('tau', stats.kendalltau(final_df.coherence.values, final_df['F1(k)'].values))

(3610, 8)
0.3513202771374517
Correlation with utility
r PearsonRResult(statistic=0.0031552517171774424, pvalue=0.8496910208966543)
rho SignificanceResult(statistic=0.03177378579319549, pvalue=0.056275776183143104)
tau SignificanceResult(statistic=0.023321505347944745, pvalue=0.05734930755441472)

Correlation with F1
r PearsonRResult(statistic=0.022717172229614127, pvalue=0.172370780838284)
rho SignificanceResult(statistic=0.04745579341758622, pvalue=0.004345619663010817)
tau SignificanceResult(statistic=0.035996507553049485, pvalue=0.004618164586599783)


In [9]:
# final_df

### QPP methods

##### Load supervised QPP results

In [46]:
def load_supervised_qpp_res(_ret, _withQV=False):
    _ret_converter = {'mt5': 'bm25_monot5', 'tct': 'colbert_E2E', 'bm25': 'bm25', 'e5': 'e5'}
        
    _suffix = 'matched' if _withQV else 'matched_withoutQV'
    _retriever = _ret_converter[_ret]
    
    _qpp_res_dict = {}
        
    # for _query_set in ['trec-dl-2019', 'trec-dl-2020']:
    #     with open(f'./supervised_results/QPP_bm25_bert-base-uncased_{_suffix}/results-{_query_set}-{_retriever}.txt') as f:
    #         for l in f:
    #             _qid, _qpp_value = l.rstrip().split('\t')
    #             _qpp_res_dict.update({_qid: float(_qpp_value)})
    #         f.close()

    with open(f'./supervised_results/QPP_bm25_bert-base-uncased_{_suffix}/results-nq_test-{_retriever}.txt') as f:
        for l in f:
            _qid, _qpp_value = l.rstrip().split('\t')
            _qpp_res_dict.update({_qid: float(_qpp_value)})
        f.close()
    
    # print(len(_qpp_res_dict)) # check the length of qppres dict
    return _qpp_res_dict 

In [47]:
bertqpp_dict = load_supervised_qpp_res(_ret)
bertqpp_qv_dict = load_supervised_qpp_res(_ret, True)

In [48]:
# final_df = eval_df.copy()
final_df_qpp = final_df.copy()
final_df_qpp['bertqpp'] = final_df_qpp.qid.apply(lambda x: float(bertqpp_dict[x]))

from scipy import stats

print('Correlation with utility')
print('r', stats.pearsonr(final_df_qpp.bertqpp.values, final_df.utility.values))
print('rho', stats.spearmanr(final_df_qpp.bertqpp.values, final_df.utility.values))
print('tau', stats.kendalltau(final_df_qpp.bertqpp.values, final_df.utility.values))

print('\nCorrelation with F1')
print('r', stats.pearsonr(final_df_qpp.bertqpp.values, final_df['F1(k)'].values))
print('rho', stats.spearmanr(final_df_qpp.bertqpp.values, final_df['F1(k)'].values))
print('tau', stats.kendalltau(final_df_qpp.bertqpp.values, final_df['F1(k)'].values))

Correlation with utility
r PearsonRResult(statistic=0.1418776316744768, pvalue=1.0818971090055897e-17)
rho SignificanceResult(statistic=0.1374470891313524, pvalue=1.087624695402813e-16)
tau SignificanceResult(statistic=0.10152764922496194, pvalue=1.292806333296747e-16)

Correlation with F1
r PearsonRResult(statistic=0.1739923185506983, pvalue=6.228547722600647e-26)
rho SignificanceResult(statistic=0.16234651046364246, pvalue=9.596892441013531e-23)
tau SignificanceResult(statistic=0.12383588603532728, pvalue=1.946820228504485e-22)


In [49]:
final_df_qpp['bertqpp_qv'] = final_df_qpp.qid.apply(lambda x: float(bertqpp_qv_dict[x]))

from scipy import stats

print('Correlation with utility')
print('r', stats.pearsonr(final_df_qpp.bertqpp_qv.values, final_df.utility.values))
print('rho', stats.spearmanr(final_df_qpp.bertqpp_qv.values, final_df.utility.values))
print('tau', stats.kendalltau(final_df_qpp.bertqpp_qv.values, final_df.utility.values))

print('\nCorrelation with F1')
print('r', stats.pearsonr(final_df_qpp.bertqpp_qv.values, final_df['F1(k)'].values))
print('rho', stats.spearmanr(final_df_qpp.bertqpp_qv.values, final_df['F1(k)'].values))
print('tau', stats.kendalltau(final_df_qpp.bertqpp_qv.values, final_df['F1(k)'].values))

Correlation with utility
r PearsonRResult(statistic=0.1350731952175633, pvalue=3.6330869482367827e-16)
rho SignificanceResult(statistic=0.13748991479475275, pvalue=1.0640081689053506e-16)
tau SignificanceResult(statistic=0.10164831410533366, pvalue=1.1903520351260445e-16)

Correlation with F1
r PearsonRResult(statistic=0.16287453018165932, pvalue=6.959130217609097e-23)
rho SignificanceResult(statistic=0.15108978088927244, pvalue=7.02502853211893e-20)
tau SignificanceResult(statistic=0.11540526552343056, pvalue=1.0745386222475346e-19)


In [50]:
# estimate their scale
print(final_df_qpp.bertqpp.apply(lambda x: math.log(x)).mean())
print(final_df_qpp.bertqpp_qv.apply(lambda x: math.log(x)).mean())
# print(final_df_qpp.coherence.apply(lambda x: math.log(x)).mean())
print(final_df_qpp.llm_prob.mean())

-1.7876695865185617
-1.6707026450075866
-2.244448742641967


#### Other QPP methods

In [51]:
import qpp_methods
qpp = qpp_methods.QPP('nq_test')

14:41:42.893 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 660.3 MiB of memory would be required.


In [52]:
import pyterrier as pt
import pyterrier_rag
import pyterrier_dr
from pyterrier_dr import E5

e5_query_encoder = E5()
e5_index = pt.Artifact.from_hf('pyterrier/ragwiki-e5.flex')

# tct_model = pyterrier_dr.TctColBert()
# # tct_index = pyterrier_dr.FlexIndex('/mnt/indices/msmarco-passage.tct-hnp.flex')
# tct_index = pyterrier_dr.FlexIndex('/mnt/indices/nq_tct_colbert_index_1.flex')

In [53]:
try:
    dense_qpp_df = pd.read_csv(f'./precomputed_qpps/{_ret}_{_k}_spatial_nq_test.csv')
except:
    dense_qpp_df = qpp.qpp_in_batch(res, 'spatial', _k, q_encoder=e5_query_encoder, _index=e5_index)
    dense_qpp_df.to_csv(f'./precomputed_qpps/{_ret}_{_k}_spatial_nq_test.csv', index=False)

In [54]:
try:
    nqc_df = pd.read_csv(f'./precomputed_qpps/{_ret}_{_k}_nqc_nq_test.csv')
except:
    nqc_df = qpp.qpp_in_batch(res, 'nqc', _k)
    nqc_df.to_csv(f'./precomputed_qpps/{_ret}_{_k}_nqc_nq_test.csv', index=False)

In [55]:
try:
    a_ratio_df = pd.read_csv(f'./precomputed_qpps/{_ret}_a_ratio_nq_test.csv')
except:
    a_ratio_df = qpp.qpp_in_batch(res, 'a_ratio', _k, q_encoder=e5_query_encoder, _index=e5_index)
    a_ratio_df.to_csv(f'./precomputed_qpps/{_ret}_{_k}_a_ratio_nq_test.csv', index=False)
    a_ratio_df.to_csv(f'./precomputed_qpps/{_ret}_a_ratio_nq_test.csv', index=False)

In [56]:
# a_ratio_df.to_csv(f'./precomputed_qpps/{_ret}_{_k}_a_ratio_nq_test.csv', index=False)
# a_ratio_df.to_csv(f'./precomputed_qpps/{_ret}_a_ratio_nq_test.csv', index=False)
# dense_qpp_df.to_csv(f'./precomputed_qpps/{_ret}_{_k}_spatial_nq_test.csv', index=False)
# nqc_df.to_csv(f'./precomputed_qpps/{_ret}_{_k}_nqc_nq_test.csv', index=False)

In [57]:
qpp_df = dense_qpp_df[['qid', 'qpp_estimate']].rename(columns={'qpp_estimate': 'spatial'})

qpp_df = qpp_df.merge(a_ratio_df[['qid', 'qpp_estimate']], on='qid')
qpp_df = qpp_df.rename(columns={'qpp_estimate': 'a_ratio'})

qpp_df = qpp_df.merge(nqc_df[['qid', 'qpp_estimate']], on='qid')
qpp_df = qpp_df.rename(columns={'qpp_estimate': 'nqc'})

In [58]:
final_df_qpp = final_df_qpp.merge(qpp_df, on='qid')

In [59]:
from scipy import stats

check_qpp = 'a_ratio'
print(f'Checking QPP method == {check_qpp}')

print('Correlation with utility')
print('r', stats.pearsonr(final_df_qpp[check_qpp].values, final_df.utility.values))
print('rho', stats.spearmanr(final_df_qpp[check_qpp].values, final_df.utility.values))
print('tau', stats.kendalltau(final_df_qpp[check_qpp].values, final_df.utility.values))

print('\nCorrelation with F1')
print('r', stats.pearsonr(final_df_qpp[check_qpp].values, final_df['F1(k)'].values))
print('rho', stats.spearmanr(final_df_qpp[check_qpp].values, final_df['F1(k)'].values))
print('tau', stats.kendalltau(final_df_qpp[check_qpp].values, final_df['F1(k)'].values))

Checking QPP method == a_ratio
Correlation with utility
r PearsonRResult(statistic=0.12746181651591879, pvalue=1.505743820956842e-14)
rho SignificanceResult(statistic=0.1187861719803562, pvalue=8.070474940777603e-13)
tau SignificanceResult(statistic=0.0873590608793767, pvalue=1.0829229528047595e-12)

Correlation with F1
r PearsonRResult(statistic=0.18328204201999254, pvalue=1.219228070520817e-28)
rho SignificanceResult(statistic=0.17402303431451627, pvalue=6.104850903491344e-26)
tau SignificanceResult(statistic=0.1336163875735432, pvalue=7.435130440398405e-26)


### combine perplexity with qpp

In [60]:
final_df_qpp['a_ratio']

0       1.042386
1       1.097404
2       1.057290
3       1.065315
4       1.032003
          ...   
3605    1.041705
3606    1.061079
3607    1.032112
3608    1.068642
3609    1.050537
Name: a_ratio, Length: 3610, dtype: float64

In [61]:
import math
##################
# mt5 best at 0.3
# bm25 best at 0.5
# e5 best at 0.3
##################

check_qpp = 'a_ratio'
print(f'Checking QPP method == {check_qpp}')

# _coeff_lambda = 0.95

_r_max = 0
# _coeff_lambda = 0.05
for _c in np.union1d(-np.arange(0.05, 1.00, 0.050), np.arange(0.05, 1.00, 0.050)):
    _r = stats.pearsonr((final_df_qpp[check_qpp].apply(lambda x: _c*(math.log(x)))+(1-_c)*final_df_qpp.llm_prob).values, final_df.utility.values)[0]
    if(_r > _r_max):
        _r_max = _r
        _coeff_lambda = _c

print(f'\ncoefficient = {round(_coeff_lambda,2)}')
    
print('\nCorrelation with utility')
print('r', stats.pearsonr((final_df_qpp[check_qpp].apply(lambda x: _coeff_lambda*(math.log(x)))+(1-_coeff_lambda)*final_df_qpp.llm_prob).values, final_df.utility.values))
print('rho', stats.spearmanr((final_df_qpp[check_qpp].apply(lambda x: _coeff_lambda*(math.log(x)))+(1-_coeff_lambda)*final_df_qpp.llm_prob).values, final_df.utility.values))
print('tau', stats.kendalltau((final_df_qpp[check_qpp].apply(lambda x: _coeff_lambda*(math.log(x)))+(1-_coeff_lambda)*final_df_qpp.llm_prob).values, final_df.utility.values))

_r_max = 0
# _coeff_lambda = 0.05
for _c in np.union1d(-np.arange(0.05, 1.00, 0.050), np.arange(0.05, 1.00, 0.050)):
    _r = stats.pearsonr((final_df_qpp[check_qpp].apply(lambda x: _c*(math.log(x)))+(1-_c)*final_df_qpp.llm_prob).values, final_df['F1(k)'].values)[0]
    if(_r > _r_max):
        _r_max = _r
        _coeff_lambda = _c

print(f'\ncoefficient = {round(_coeff_lambda,2)}')

print('\nCorrelation with F1')
print('r', stats.pearsonr((final_df_qpp[check_qpp].apply(lambda x: _coeff_lambda*(math.log(x)))+(1-_coeff_lambda)*final_df_qpp.llm_prob).values, final_df['F1(k)'].values))
print('rho', stats.spearmanr((final_df_qpp[check_qpp].apply(lambda x: _coeff_lambda*(math.log(x)))+(1-_coeff_lambda)*final_df_qpp.llm_prob).values, final_df['F1(k)'].values))
print('tau', stats.kendalltau((final_df_qpp[check_qpp].apply(lambda x: _coeff_lambda*(math.log(x)))+(1-_coeff_lambda)*final_df_qpp.llm_prob).values, final_df['F1(k)'].values))

Checking QPP method == a_ratio

coefficient = 0.95

Correlation with utility
r PearsonRResult(statistic=0.11211895911413204, pvalue=1.4239226750906787e-11)
rho SignificanceResult(statistic=0.11547423415213492, pvalue=3.4280092091445697e-12)
tau SignificanceResult(statistic=0.08504563284220543, pvalue=4.178877675318281e-12)

coefficient = 0.95

Correlation with F1
r PearsonRResult(statistic=0.19313562702023312, pvalue=1.1245037645225472e-31)
rho SignificanceResult(statistic=0.18502551005268184, pvalue=3.641562576297849e-29)
tau SignificanceResult(statistic=0.14131998190430878, pvalue=9.989672065756409e-29)


In [66]:
final_df_qpp_A

,F1(0),F1(k),EM(0),EM(k),qid,utility,llm_prob,coherence,bertqpp,bertqpp_qv,spatial,a_ratio,nqc
0,1.000000,0.800000,1.0,0.0,test_0,-0.200000,-1.234375,0.079970,0.031214,0.066502,2999.302002,1.042386,0.000718
1,0.000000,0.333333,0.0,0.0,test_1,0.333333,-2.677734,0.006003,0.226111,0.327250,2960.055176,1.097404,0.000420
2,0.000000,0.000000,0.0,0.0,test_2,0.000000,-2.427734,0.005563,0.244679,0.482571,2849.471436,1.057290,0.000101
3,0.000000,0.000000,0.0,0.0,test_3,0.000000,-2.322266,0.068849,0.348521,0.242520,2927.418945,1.065315,0.000349
4,0.000000,0.000000,0.0,0.0,test_4,0.000000,-2.878906,0.003028,0.061742,0.030930,2734.755615,1.032003,0.000143
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0.333333,0.666667,0.0,0.0,test_995,0.333333,-2.169922,0.017047,0.016848,0.014885,2927.694092,1.054598,0.000763
996,0.000000,0.000000,0.0,0.0,test_996,0.000000,-1.731445,0.110587,0.010278,0.014888,3007.695312,1.075530,0.001317
997,0.000000,0.000000,0.0,0.0,test_997,0.000000,-2.443359,0.004857,0.283126,0.396579,2932.395996,1.057166,0.000049
998,0.000000,0.666667,0.0,0.0,test_998,0.666667,-1.688477,0.079621,0.277895,0.275980,2956.338867,1.062547,0.000350


In [81]:
import math
##################
# mt5 best at 0.3
# bm25 best at 0.5
# e5 best at 0.3
##################

check_qpp = 'a_ratio'
print(f'Checking QPP method == {check_qpp}')

A_size = 3000
final_df_qpp_A= final_df_qpp.iloc[:A_size]

_r_max = 0
# _coeff_lambda = 0.05
for _c in np.union1d(-np.arange(0.05, 1.00, 0.050), np.arange(0.05, 1.00, 0.050)):
    _r = stats.pearsonr((final_df_qpp_A[check_qpp].apply(lambda x: _c*(math.log(x)))+(1-_c)*final_df_qpp_A.llm_prob).values, final_df.utility.values[:A_size])[0]
    if(_r > _r_max):
        _r_max = _r
        _coeff_lambda = _c

print(f'\ncoefficient = {round(_coeff_lambda,2)}')
    
print('\nCorrelation with utility')
print('r', stats.pearsonr((final_df_qpp_A[check_qpp].apply(lambda x: _coeff_lambda*(math.log(x)))+(1-_coeff_lambda)*final_df_qpp_A.llm_prob).values, final_df.utility.values[:A_size]))
print('rho', stats.spearmanr((final_df_qpp_A[check_qpp].apply(lambda x: _coeff_lambda*(math.log(x)))+(1-_coeff_lambda)*final_df_qpp_A.llm_prob).values, final_df.utility.values[:A_size]))
print('tau', stats.kendalltau((final_df_qpp_A[check_qpp].apply(lambda x: _coeff_lambda*(math.log(x)))+(1-_coeff_lambda)*final_df_qpp_A.llm_prob).values, final_df.utility.values[:A_size]))

_r_max = 0
# _coeff_lambda = 0.05
for _c in np.union1d(-np.arange(0.05, 1.00, 0.050), np.arange(0.05, 1.00, 0.050)):
    _r = stats.pearsonr((final_df_qpp_A[check_qpp].apply(lambda x: _c*(math.log(x)))+(1-_c)*final_df_qpp_A.llm_prob).values, final_df['F1(k)'].values[:A_size])[0]
    if(_r > _r_max):
        _r_max = _r
        _coeff_lambda = _c

print(f'\ncoefficient = {round(_coeff_lambda,2)}')

print('\nCorrelation with F1')
print('r', stats.pearsonr((final_df_qpp_A[check_qpp].apply(lambda x: _coeff_lambda*(math.log(x)))+(1-_coeff_lambda)*final_df_qpp_A.llm_prob).values, final_df['F1(k)'].values[:A_size]))
print('rho', stats.spearmanr((final_df_qpp_A[check_qpp].apply(lambda x: _coeff_lambda*(math.log(x)))+(1-_coeff_lambda)*final_df_qpp_A.llm_prob).values, final_df['F1(k)'].values[:A_size]))
print('tau', stats.kendalltau((final_df_qpp_A[check_qpp].apply(lambda x: _coeff_lambda*(math.log(x)))+(1-_coeff_lambda)*final_df_qpp_A.llm_prob).values, final_df['F1(k)'].values[:A_size]))

Checking QPP method == a_ratio

coefficient = 0.95

Correlation with utility
r PearsonRResult(statistic=0.1225475455307252, pvalue=1.6414852689389894e-11)
rho SignificanceResult(statistic=0.1276356222167647, pvalue=2.2718221334036537e-12)
tau SignificanceResult(statistic=0.09376840681015229, pvalue=3.1275284354635944e-12)

coefficient = 0.95

Correlation with F1
r PearsonRResult(statistic=0.2122185549028213, pvalue=6.818878688415934e-32)
rho SignificanceResult(statistic=0.20556248729716634, pvalue=5.498909790320923e-30)
tau SignificanceResult(statistic=0.15699934555618522, pvalue=1.9450993730822355e-29)


In [183]:
final_df_qpp.spatial.apply(lambda x: math.log(x)).mean()

7.927690081184484

In [184]:
final_df_qpp.llm_prob.mean()

-2.244448742641967

### combine coherence with qpp

In [25]:
import math

_coeff_lambda = 0.6

print('Correlation with utility')
print('r', stats.pearsonr((final_df_qpp.bertqpp.apply(lambda x: _coeff_lambda*math.log(x))+final_df_qpp.coherence.apply(lambda x: (1-_coeff_lambda)*math.log(x)).values), final_df.utility.values))
print('rho', stats.spearmanr((final_df_qpp.bertqpp.apply(lambda x: _coeff_lambda*math.log(x))+final_df_qpp.coherence.apply(lambda x: (1-_coeff_lambda)*math.log(x)).values), final_df.utility.values))
print('tau', stats.kendalltau((final_df_qpp.bertqpp.apply(lambda x: _coeff_lambda*math.log(x))+final_df_qpp.coherence.apply(lambda x: (1-_coeff_lambda)*math.log(x)).values), final_df.utility.values))

print('\nCorrelation with F1')
print('r', stats.pearsonr((final_df_qpp.bertqpp.apply(lambda x: _coeff_lambda*math.log(x))+final_df_qpp.coherence.apply(lambda x: (1-_coeff_lambda)*math.log(x)).values), final_df['F1(k)'].values))
print('rho', stats.spearmanr((final_df_qpp.bertqpp.apply(lambda x: _coeff_lambda*math.log(x))+final_df_qpp.coherence.apply(lambda x: (1-_coeff_lambda)*math.log(x)).values), final_df['F1(k)'].values))
print('tau', stats.kendalltau((final_df_qpp.bertqpp.apply(lambda x: _coeff_lambda*math.log(x))+final_df_qpp.coherence.apply(lambda x: (1-_coeff_lambda)*math.log(x)).values), final_df['F1(k)'].values))

Correlation with utility
r PearsonRResult(statistic=0.08770233877998661, pvalue=1.307506179091683e-07)
rho SignificanceResult(statistic=0.10035203204675608, pvalue=1.5160409965358174e-09)
tau SignificanceResult(statistic=0.07382684220909413, pvalue=1.3138397022329281e-09)

Correlation with F1
r PearsonRResult(statistic=0.13060237303917313, pvalue=3.3249169803020163e-15)
rho SignificanceResult(statistic=0.13376413694253356, pvalue=7.001300634796929e-16)
tau SignificanceResult(statistic=0.10093464814561576, pvalue=9.741029667417055e-16)


In [26]:
# spatial_qpp_df = qpp.qpp_in_batch(res[res.qid.isin(res.qid.unique())], 'spatial', _k, tct_model, tct_index)

In [27]:
# spatial_qpp_df.to_csv(f'./qpp_cache_dense_{_ret}_{_k}.csv', index=False)

In [29]:
# qpp_df = qpp.qpp_in_batch(res, 'nqc', _k, tct_model, tct_index)

In [ ]:
final_df_1 = final_df.merge(qpp_df, on='qid')
final_df_1['coh_spatial'] = final_df_1['coherence'].apply(lambda x: math.log(1+x)) + final_df_1['qpp_estimate'].apply(lambda x: math.log(1+x))
final_df_1.shape

In [ ]:
from scipy import stats

print('Correlation with utility')
print('r', stats.pearsonr(final_df_1.qpp_estimate.values, final_df_1.utility.values))
print('rho', stats.spearmanr(final_df_1.qpp_estimate.values, final_df_1.utility.values))
print('tau', stats.kendalltau(final_df_1.qpp_estimate.values, final_df_1.utility.values))

print('\nCorrelation with F1')
print('r', stats.pearsonr(final_df_1.qpp_estimate.values, final_df_1['F1(k)'].values))
print('rho', stats.spearmanr(final_df_1.qpp_estimate.values, final_df_1['F1(k)'].values))
print('tau', stats.kendalltau(final_df_1.qpp_estimate.values, final_df_1['F1(k)'].values))

In [ ]:
from scipy import stats

print('Correlation with utility')
print('r', stats.pearsonr(final_df_1.coh_spatial.values, final_df_1.utility.values))
print('rho', stats.spearmanr(final_df_1.coh_spatial.values, final_df_1.utility.values))
print('tau', stats.kendalltau(final_df_1.coh_spatial.values, final_df_1.utility.values))

print('\nCorrelation with F1')
print('r', stats.pearsonr(final_df_1.coh_spatial.values, final_df_1['F1(k)'].values))
print('rho', stats.spearmanr(final_df_1.coh_spatial.values, final_df_1['F1(k)'].values))
print('tau', stats.kendalltau(final_df_1.coh_spatial.values, final_df_1['F1(k)'].values))

In [ ]:
import pyterrier as pt

sparse_index = pt.Artifact.from_hf('pyterrier/ragwiki-terrier')

In [ ]:
nq_index_ref = pt.IndexFactory.of('/mnt/indices/BEIR/nq/nq_sparseIndex')

nq_index_ref.getCollectionStatistics().getNumberOfDocuments()

In [ ]:
index_path ='/mnt/indices/msmarco-passage.terrier/'
index_ref = pt.IndexRef.of(index_path)
index = pt.IndexFactory.of(index_ref)
DOC_NUM = index.getCollectionStatistics().getNumberOfDocuments()

In [ ]:
type(sparse_index.path)

In [ ]:
import torch

torch.cuda.empty_cache()